# VeriRAG Evaluation Workflow

This notebook evaluates the saved VeriRAG model on the held-out test split and generates metrics, visualizations, and error analysis artifacts for research reporting.

## 1. Import Libraries

In [ ]:
import csv
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_curve,
    auc,
    precision_recall_curve
)
from sklearn.preprocessing import label_binarize
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from config import TrainConfig, id2label
from dataset import RAGTruthDataset, load_and_split_ragtruth

## 2. Load Test Split and Best Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

_, _, test_samples = load_and_split_ragtruth(TrainConfig.DATASET_PATH)
checkpoint_dir = Path(TrainConfig.OUTPUT_DIR)
output_dir = checkpoint_dir
output_dir.mkdir(parents=True, exist_ok=True)

print('====================================================')
print('Evaluation Configuration')
print('====================================================')
print(f'Checkpoint:    {checkpoint_dir}')
print(f'Dataset:       {TrainConfig.DATASET_PATH}')
print(f'Device:        {device}')
print(f'Test Samples:  {len(test_samples)}')
print(f'Batch Size:    {TrainConfig.EVAL_BATCH_SIZE}')
print('====================================================')

tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir).to(device)

test_dataset = RAGTruthDataset(test_samples, tokenizer, max_length=TrainConfig.MAX_SEQ_LENGTH)
test_loader = DataLoader(test_dataset, batch_size=TrainConfig.EVAL_BATCH_SIZE, shuffle=False)
print(f'Loaded saved model from {checkpoint_dir}')

## 3. Generate Predictions

In [ ]:
all_targets = []
all_preds = []
all_probs = []
latencies = []

model.eval()
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['labels'].cpu().numpy()

        t0 = time.time()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        t1 = time.time()

        logits = outputs.logits
        probs = F.softmax(logits, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)

        all_targets.extend(targets.tolist())
        all_preds.extend(preds.tolist())
        all_probs.extend(probs.tolist())
        latencies.extend([((t1 - t0) * 1000.0) / len(targets)] * len(targets))

all_targets = np.array(all_targets)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

print('Prediction generation complete.')

## 4. Evaluation Metrics

In [ ]:
labels = list(id2label.keys())
target_names = [id2label[i] for i in labels]

accuracy = accuracy_score(all_targets, all_preds)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    all_targets, all_preds, average='macro', zero_division=0
)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
    all_targets, all_preds, average='weighted', zero_division=0
)
cm = confusion_matrix(all_targets, all_preds, labels=labels)
avg_latency = float(np.mean(latencies))
median_latency = float(np.median(latencies))
max_latency = float(np.max(latencies))
min_latency = float(np.min(latencies))
report = classification_report(
    all_targets, all_preds, labels=labels, target_names=target_names,
    zero_division=0, output_dict=True
)

metrics = {
    'accuracy': float(accuracy),
    'precision_macro': float(precision_macro),
    'recall_macro': float(recall_macro),
    'f1_macro': float(f1_macro),
    'precision_weighted': float(precision_weighted),
    'recall_weighted': float(recall_weighted),
    'f1_weighted': float(f1_weighted),
    'avg_latency_ms': avg_latency,
    'median_latency_ms': median_latency,
    'max_latency_ms': max_latency,
    'min_latency_ms': min_latency,
}

print('====================================================')
print('Evaluation Summary')
print('====================================================')
print(f'Accuracy:         {metrics["accuracy"]:.4f}')
print(f'Macro Precision:  {metrics["precision_macro"]:.4f}')
print(f'Macro Recall:     {metrics["recall_macro"]:.4f}')
print(f'Macro F1:         {metrics["f1_macro"]:.4f}')
print(f'Weighted F1:      {metrics["f1_weighted"]:.4f}')
print(f'Average Latency:  {metrics["avg_latency_ms"]:.2f} ms')
print(f'Median Latency:   {metrics["median_latency_ms"]:.2f} ms')
print(f'Maximum Latency:  {metrics["max_latency_ms"]:.2f} ms')
print(f'Minimum Latency:  {metrics["min_latency_ms"]:.2f} ms')
print('====================================================')

## 5. Visualizations

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title('Confusion Matrix')
plt.colorbar()
plt.xticks(range(len(labels)), target_names, rotation=45, ha='right')
plt.yticks(range(len(labels)), target_names)
for i in range(len(labels)):
    for j in range(len(labels)):
        plt.text(j, i, cm[i, j], ha='center', va='center', color='black')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig(output_dir / 'confusion_matrix.png', dpi=150)
plt.show()

plt.figure(figsize=(8, 5))
sample_counts = [np.sum(all_preds == label) for label in labels]
plt.bar(target_names, sample_counts, color='tab:purple')
plt.title('Prediction Distribution by Class')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(output_dir / 'prediction_distribution.png', dpi=150)
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(latencies, bins=20, color='tab:green', edgecolor='black')
plt.title('Inference Latency Distribution (ms/sample)')
plt.xlabel('Latency (ms)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(output_dir / 'latency_histogram.png', dpi=150)
plt.show()

## 6. ROC and Precision-Recall Curves

In [ ]:
binarized_targets = label_binarize(all_targets, classes=labels)

plt.figure(figsize=(10, 8))
for idx, label_name in enumerate(target_names):
    fpr, tpr, _ = roc_curve(binarized_targets[:, idx], all_probs[:, idx])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{label_name} (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)
plt.title('One-vs-Rest ROC Curves')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.savefig(output_dir / 'roc_curves.png', dpi=150)
plt.show()

plt.figure(figsize=(10, 8))
for idx, label_name in enumerate(target_names):
    precision, recall, _ = precision_recall_curve(binarized_targets[:, idx], all_probs[:, idx])
    plt.plot(recall, precision, label=label_name)
plt.title('One-vs-Rest Precision-Recall Curves')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend(loc='lower left')
plt.grid(True)
plt.tight_layout()
plt.savefig(output_dir / 'precision_recall_curves.png', dpi=150)
plt.show()

## 7. Error Analysis

In [ ]:
results = []
for sample, true_label, pred_label, probs in zip(test_samples, all_targets, all_preds, all_probs):
    results.append({
        'id': sample.get('id', ''),
        'context': sample.get('context', ''),
        'claim': sample.get('claim', ''),
        'true_label': id2label[true_label],
        'pred_label': id2label[pred_label],
        'confidence': float(probs[pred_label]),
        'correct': true_label == pred_label,
    })

correct_samples = [row for row in results if row['correct']]
incorrect_samples = [row for row in results if not row['correct']]

# Rank misclassified samples by confidence (most confident errors first)
top_misclassified = sorted(incorrect_samples, key=lambda row: row['confidence'], reverse=True)
highest_confidence_wrong = top_misclassified[:5]

# Rank correct predictions by confidence (least confident correct first)
lowest_confidence_correct = sorted(correct_samples, key=lambda row: row['confidence'])[:5]

print(f'Correct Predictions:   {len(correct_samples)}')
print(f'Incorrect Predictions: {len(incorrect_samples)}')

print('\nTop Misclassified Samples:')
for row in top_misclassified[:5]:
    print(f"- True: {row['true_label']} | Pred: {row['pred_label']} | Confidence: {row['confidence']:.3f}")
    print(f"  Claim: {row['claim']}")
    print(f"  Context: {row['context'][:180].replace(chr(10), ' ')} ...")

print('\nHighest Confidence Wrong Predictions:')
for row in highest_confidence_wrong:
    print(f"- True: {row['true_label']} | Pred: {row['pred_label']} | Confidence: {row['confidence']:.3f}")
    print(f"  Claim: {row['claim']}")

print('\nLowest Confidence Correct Predictions:')
for row in lowest_confidence_correct:
    print(f"- True: {row['true_label']} | Pred: {row['pred_label']} | Confidence: {row['confidence']:.3f}")
    print(f"  Claim: {row['claim']}")

## 8. Export Metrics and Reports

In [ ]:
with open(output_dir / 'evaluation_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

experiment_summary = {
    'accuracy': metrics['accuracy'],
    'precision': metrics['precision_macro'],
    'recall': metrics['recall_macro'],
    'macro_f1': metrics['f1_macro'],
    'weighted_f1': metrics['f1_weighted'],
    'inference_latency_ms': {
        'average': metrics['avg_latency_ms'],
        'median': metrics['median_latency_ms'],
        'maximum': metrics['max_latency_ms'],
        'minimum': metrics['min_latency_ms'],
    },
    'checkpoint_used': str(checkpoint_dir),
    'dataset': str(TrainConfig.DATASET_PATH),
    'evaluation_date': time.strftime('%Y-%m-%d %H:%M:%S'),
}

with open(output_dir / 'experiment_summary.json', 'w', encoding='utf-8') as f:
    json.dump(experiment_summary, f, indent=2)

with open(output_dir / 'classification_report.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['label', 'precision', 'recall', 'f1-score', 'support'])
    for label_name in target_names:
        label_report = report[label_name]
        writer.writerow([
            label_name,
            label_report['precision'],
            label_report['recall'],
            label_report['f1-score'],
            label_report['support'],
        ])
    for avg_label in ['macro avg', 'weighted avg']:
        avg_report = report[avg_label]
        writer.writerow([
            avg_label,
            avg_report['precision'],
            avg_report['recall'],
            avg_report['f1-score'],
            avg_report['support'],
        ])
    writer.writerow(['accuracy', report['accuracy'], '', '', report['macro avg']['support']])

with open(output_dir / 'prediction_results.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Context', 'Claim', 'Ground Truth', 'Prediction', 'Confidence', 'Correct / Incorrect'])
    for row in results:
        writer.writerow([
            row['context'],
            row['claim'],
            row['true_label'],
            row['pred_label'],
            f"{row['confidence']:.4f}",
            'Correct' if row['correct'] else 'Incorrect',
        ])

print(f'Saved evaluation artifacts to {output_dir}')
print('- evaluation_metrics.json')
print('- experiment_summary.json')
print('- classification_report.csv')
print('- prediction_results.csv')